In [8]:
S3_ROOT = "s3://smart-park-seattle/parking_v2/" # v2
YEAR = 2023

FEAT_TOPK = 16
TIN = 12
BATCH_SIZE = 8

In [9]:
from parking_processing.inference.predict_mvstgcn import PredictCfg, run_predict_mvstgcn
import json
from parking_processing.utils.s3 import s3_client, parse_s3_uri, s3_get_text

# load splits to know which weeks to predict
s3 = s3_client()
bucket, prefix = parse_s3_uri(S3_ROOT)
splits = json.loads(s3_get_text(s3, bucket, f"{prefix}splits/year={YEAR}/splits.json"))
weeks = splits["train_weeks"] + splits["val_weeks"] + splits["test_weeks"]

In [2]:
# Create fallback prediction for holidays and sundays parking
from parking_processing.fallback.sat_profile import SatProfileCfg, run_build_sat_profile

profile_uri = run_build_sat_profile(SatProfileCfg(
    s3_root=S3_ROOT,
    year_ref=2023,
    target="y_15",
    tz_local="America/Los_Angeles",
))
print("profile:", profile_uri)

profile: s3://smart-park-seattle/parking_v2/meta/sat_profile/year_ref=2023/target=y_15/profile.csv.gz


In [4]:
from parking_processing.fallback.predict_fallback import FallbackCfg, run_fallback_for_weeks

run_fallback_for_weeks(FallbackCfg(
    s3_root=S3_ROOT,
    year=YEAR,
    target="y_15",
    sat_profile_s3=profile_uri,
    free_days_key="meta/free_days.csv",
    model_name="fallback",
), weeks)

[ok] week=2023-01-02 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_15/week=2023-01-02/pred.csv.gz
[ok] week=2023-01-09 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_15/week=2023-01-09/pred.csv.gz
[ok] week=2023-01-16 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_15/week=2023-01-16/pred.csv.gz
[ok] week=2023-01-23 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_15/week=2023-01-23/pred.csv.gz
[ok] week=2023-01-30 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_15/week=2023-01-30/pred.csv.gz
[ok] week=2023-02-06 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_15/week=2023-02-06/pred.csv.gz
[ok] week=2023-02-13 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_15/week=2023-02-13/pred.csv.gz
[ok] week=2023-02-20 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_15/week=2023-02-20/pred.csv.gz
[ok] wee

In [12]:
profile_uri = run_build_sat_profile(SatProfileCfg(
    s3_root=S3_ROOT,
    year_ref=2023,
    target="y_30",
    tz_local="America/Los_Angeles",
))
print("profile:", profile_uri)

profile: s3://smart-park-seattle/parking_v2/meta/sat_profile/year_ref=2023/target=y_30/profile.csv.gz


In [13]:
from parking_processing.fallback.predict_fallback import FallbackCfg, run_fallback_for_weeks
run_fallback_for_weeks(FallbackCfg(
    s3_root=S3_ROOT,
    year=YEAR,
    target="y_30",
    sat_profile_s3=profile_uri,
    free_days_key="meta/free_days.csv",
    model_name="fallback",
), weeks)

[ok] week=2023-01-02 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_30/week=2023-01-02/pred.csv.gz
[ok] week=2023-01-09 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_30/week=2023-01-09/pred.csv.gz
[ok] week=2023-01-16 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_30/week=2023-01-16/pred.csv.gz
[ok] week=2023-01-23 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_30/week=2023-01-23/pred.csv.gz
[ok] week=2023-01-30 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_30/week=2023-01-30/pred.csv.gz
[ok] week=2023-02-06 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_30/week=2023-02-06/pred.csv.gz
[ok] week=2023-02-13 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_30/week=2023-02-13/pred.csv.gz
[ok] week=2023-02-20 -> s3://smart-park-seattle/parking_v2/preds/fallback/year=2023/target=y_30/week=2023-02-20/pred.csv.gz
[ok] wee

Task completed: y-15 and y-30